In [1]:
import bw2data as bd
import bw2io as bi
import bw2calc as bc
import bw2analyzer as bwa
import polars as pl 
from tqdm import tqdm

bd.projects.set_current("2026-01-13-ecoinvent-3.11-cutoff")

09:51:29+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [3]:
df= pl.read_parquet("/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/Sensibilite/2026-6-14-0h12_OAT_LSA_Logement_collectif.parquet")

In [5]:
df

archetype,type,procédé_concerné,indicator_or_ef,unit,EF_type,EF_categories,xj_tilde,xj_modif,y_tilde,y_modif,lot,sous_lot,delta_y,delta_xj,sensibilite,elasticite
str,str,str,str,str,str,str,f64,f64,f64,f64,str,str,f64,f64,f64,f64
"""Logement_collectif""","""inventory""","""diffusion d’air_terminaux pass…","""Flurochloridone""","""kilogram""","""emission""","""air :: non-urban air or from h…",18188.140625,20006.954688,6.7779e-12,6.7779e-12,"""8""","""4""",3.2034e-17,1818.814063,1.7613e-20,0.000047
"""Logement_collectif""","""inventory""","""diffusion d’air_terminaux pass…","""Phenol""","""kilogram""","""emission""","""water :: ocean""",18188.140625,20006.954688,0.000107,0.000107,"""8""","""4""",4.3239e-9,1818.814063,2.3773e-12,0.000402
"""Logement_collectif""","""inventory""","""diffusion d’air_terminaux pass…","""Solids, inorganic""","""kilogram""","""emission""","""water :: ground-""",18188.140625,20006.954688,0.126815,0.126845,"""8""","""4""",0.00003,1818.814063,1.6281e-8,0.002335
"""Logement_collectif""","""inventory""","""diffusion d’air_terminaux pass…","""Prothioconazole""","""kilogram""","""emission""","""water :: surface water""",18188.140625,20006.954688,1.0154e-14,1.0154e-14,"""8""","""4""",2.4855e-20,1818.814063,1.3666e-23,0.000024
"""Logement_collectif""","""inventory""","""diffusion d’air_terminaux pass…","""Sulfadiazine""","""kilogram""","""emission""","""water""",18188.140625,20006.954688,4.2993e-10,4.2993e-10,"""8""","""4""",4.4164e-15,1818.814063,2.4282e-18,0.000103
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Logement_collectif""","""impact""","""menuiseries intérieures et ext…","""('ecoinvent-3.11', 'EF v3.1', …","""kg Sb-Eq""","""""","""""",1755.256104,1930.781714,0.024368,0.024368,"""8""","""4""",4.9423e-8,175.52561,2.8157e-10,0.00002
"""Logement_collectif""","""impact""","""menuiseries intérieures et ext…","""('ecoinvent-3.11', 'EF v3.1', …","""kg CFC-11-Eq""","""""","""""",1755.256104,1930.781714,0.000179,0.000179,"""8""","""4""",4.5366e-10,175.52561,2.5846e-12,0.000025
"""Logement_collectif""","""impact""","""menuiseries intérieures et ext…","""('ecoinvent-3.11', 'EF v3.1', …","""disease incidence""","""""","""""",1755.256104,1930.781714,0.000093,0.000093,"""8""","""4""",4.9210e-9,175.52561,2.8036e-11,0.000526


In [12]:
from pathlib import Path

input_folder = Path("/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/data/processed/Sensibilite")
fichiers = list(input_folder.glob("*.parquet"))

output_folder = "/home/thibault.chevilliet@enpc.fr/Documents/2601_archetypes_re2020/reports/tables/Sensibilite/"

In [21]:
for f in fichiers :
    print("_".join(str(f).split("/")[-1].split(".")[0].split("_")[3:]))

Maison_individuelle_ou_accolée
Bureau
Enseignement_primaire
Logement_collectif


In [26]:
nb_firsts = 50
col = "elasticite"
filtre = pl.col("indicator_or_ef").str.contains("'climate change',")

for f in fichiers :
    usage = "_".join(str(f).split("/")[-1].split(".")[0].split("_")[3:])
    output_path = f"{output_folder}/OAT_LSA_{nb_firsts}_{col}_{usage}.xlsx"
    df = pl.read_parquet(f)
    df.filter(
        filtre
        ).group_by(
            pl.col("procédé_concerné")
            ).agg(
                pl.col(col).max()
                ).sort(
                    by = col,descending=True
                    )[:nb_firsts].write_excel(output_path)


In [10]:
df.filter(pl.col("indicator_or_ef").str.contains("'climate change',")).group_by(pl.col("procédé_concerné")).agg(pl.col("sensibilite").max()).sort(by = "sensibilite",descending=True)[:50].write_excel(output_path)

In [26]:
df.filter(pl.col("indicator_or_ef").str.contains("'climate change',"))["indicator_or_ef"].unique().to_list()

["('ecoinvent-3.11', 'EF v3.1', 'climate change', 'global warming potential (GWP100)')"]

In [42]:
bd.databases

Databases dictionary with 11 object(s):
	2026-3-19-11h18_processes_RE2020
	2026-4-21_consumptions_processes
	2026-4-22-15h1_buildings_RE2020
	2026-4-22-16h25_archetypes_RE2020_BB
	2026-4-22-16h7_buildings_cons_RE2020
	2026-4-23-18h10_archetypes_RE2020_PRB
	2026-5-7-20h29_archetypes_RE2020_MIB
	2026-5-7-20h29_bernoulli_RE2020
	ecoinvent-3.11-biosphere
	ecoinvent-3.11-cutoff
	pr_based_arch_alamain

In [43]:
biodb = bd.Database("ecoinvent-3.11-biosphere")

In [58]:
b= biodb.random()

In [59]:
b.as_dict()

{'categories': ('natural resource', 'land'),
 'code': '82f2476e-988b-4d7a-bde6-5da9337d1f65',
 'CAS number': None,
 'synonyms': [],
 'name': 'Transformation, from pasture, man made, extensive',
 'database': 'ecoinvent-3.11-biosphere',
 'unit': 'square meter',
 'type': 'natural resource',
 'id': 269493372944998405}

In [66]:
m= bd.methods.random()

In [67]:
m

('ecoinvent-3.11',
 'EN15804+A2 - Indicators describing resource use',
 'total use of non-renewable primary energy resources (primary energy and primary energy resources used as raw materials)',
 'PENRT')

In [68]:
bd.methods[m]["unit"]

'MJ'